# 03 — Expected loss & portfolio concentration
Narrative around `sql/06_marts.sql` and `app/sim_core.py`. Where is risk concentrated,
and what does the policy trade-off look like?

Prereq: `python run_pipeline.py all` has been run.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
np.seterr(all="ignore")
from src.config import load
from src.db import connect
from app.sim_core import approved_metrics, profit_curve, best_policy, money

cfg = load()
con = connect(cfg["paths"]["duckdb"], read_only=True)

In [2]:
el = con.execute("SELECT * FROM mart_loan_el").fetchdf()
print(f"{len(el):,} loans")
el[["pd_hat", "ead", "expected_loss"]].describe().round(2)

640,919 loans


,pd_hat,ead,expected_loss
count,640919.00,640919.00,640919.00
mean,0.13,12726.92,739.78
std,0.08,7882.09,655.09
min,0.00,1000.00,0.00
25%,0.08,7000.00,307.17
50%,0.12,10000.00,551.17
75%,0.18,16850.00,959.54
max,0.71,35000.00,11185.98


## Concentration: share of exposure vs share of expected loss
The memo headline: "segment X is A% of exposure but B% of expected loss".

In [3]:
base = con.execute("SELECT * FROM mart_simulator_base").fetchdf()

def concentration(col):
    g = base.groupby(col).agg(n=("loan_id", "size"), exposure=("loan_amnt", "sum"),
                              el=("expected_loss", "sum"), avg_pd=("pd_hat", "mean"),
                              obs_dr=("default_flag", "mean"))
    g["exposure_share"] = g.exposure / g.exposure.sum()
    g["el_share"] = g.el / g.el.sum()
    g["el_per_exposure"] = (g.el_share / g.exposure_share).round(2)
    return g.sort_values("el_per_exposure", ascending=False).round(3)

concentration("lc_grade")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
lc_grade,,,,,,,,
G,546,6.719700e+06,8.687080e+05,0.288,0.41,0.001,0.002,2.22
F,4416,4.369788e+07,5.154547e+06,0.250,0.338,0.005,0.011,2.03
E,22124,2.658408e+08,2.820403e+07,0.229,0.294,0.033,0.059,1.83
D,77437,9.282415e+08,8.404713e+07,0.200,0.239,0.114,0.177,1.56
C,169208,2.045114e+09,1.503153e+08,0.166,0.182,0.251,0.317,1.26
B,221171,2.775407e+09,1.444266e+08,0.119,0.113,0.340,0.305,0.90
A,146017,2.092035e+09,6.111981e+07,0.068,0.054,0.256,0.129,0.50


In [4]:
concentration("purpose")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
purpose,,,,,,,,
small_business,6714,9.340000e+07,8.918340e+06,0.208,0.224,0.011,0.019,1.64
renewable_energy,431,3.807175e+06,2.764474e+05,0.154,0.197,0.000,0.001,1.25
other,33517,2.810280e+08,1.942972e+07,0.154,0.164,0.034,0.041,1.19
house,2516,3.203972e+07,2.191915e+06,0.158,0.188,0.004,0.005,1.18
medical,7036,5.214218e+07,3.542559e+06,0.152,0.173,0.006,0.007,1.17
moving,4491,3.077628e+07,2.086869e+06,0.151,0.196,0.004,0.004,1.17
vacation,4310,2.430618e+07,1.641789e+06,0.145,0.161,0.003,0.003,1.16
wedding,1174,1.141072e+07,7.262587e+05,0.145,0.127,0.001,0.002,1.09
educational,1,2.200000e+03,1.359140e+02,0.137,0.0,0.000,0.000,1.06


In [5]:
concentration("vintage_year")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
vintage_year,,,,,,,,
2012-01-01,43470,5.077991e+08,3.014242e+07,0.134,0.136,0.062,0.064,1.02
2015-01-01,283173,3.626461e+09,2.142072e+08,0.137,0.149,0.445,0.452,1.02
2014-01-01,162570,2.046041e+09,1.193849e+08,0.136,0.137,0.251,0.252,1.00
2016-01-01,51284,7.046628e+08,4.074566e+07,0.133,0.148,0.086,0.086,0.99
2013-01-01,100422,1.272091e+09,6.965593e+07,0.127,0.123,0.156,0.147,0.94


## Policy trade-off — the same math the simulator uses (`app/sim_core.py`)

In [6]:
SIM = cfg["simulator"]
args = dict(fico_min=SIM["default_min_fico"], cost_of_funds=SIM["cost_of_funds"],
            horizon=SIM["horizon_years"], amort_factor=SIM["amort_factor"],
            servicing_cost=SIM["servicing_cost"])
rows = []
for name, cut in [("Conservative", 0.08), ("Current", 0.15), ("Growth", 0.20)]:
    m = approved_metrics(base, cut, **args)
    rows.append({"policy": name, "pd_cut": cut, **{k: m[k] for k in
                 ("approval_rate", "volume", "exp_default_rate", "exp_loss", "exp_profit")}})
curve = profit_curve(base, args["fico_min"], args["cost_of_funds"], args["horizon"],
                     n_points=120, amort_factor=args["amort_factor"], servicing_cost=args["servicing_cost"])
bp = best_policy(curve)
m = approved_metrics(base, bp["pd_cut"], **args)
rows.append({"policy": "Profit-max", "pd_cut": round(bp["pd_cut"], 3), **{k: m[k] for k in
             ("approval_rate", "volume", "exp_default_rate", "exp_loss", "exp_profit")}})
pd.DataFrame(rows).assign(
    volume=lambda d: d.volume.map(money), exp_loss=lambda d: d.exp_loss.map(money),
    exp_profit=lambda d: d.exp_profit.map(money),
    approval_rate=lambda d: (d.approval_rate * 100).round(1),
    exp_default_rate=lambda d: (d.exp_default_rate * 100).round(2))

,policy,pd_cut,approval_rate,volume,exp_default_rate,exp_loss,exp_profit
0,Conservative,0.080,27.0,$2.5B,5.21,$58.2M,$17.8M
1,Current,0.150,64.0,$5.4B,8.79,$207.4M,$54.0M
2,Growth,0.200,80.8,$6.7B,10.55,$304.7M,$54.0M
3,Profit-max,0.174,73.2,$6.1B,9.71,$257.0M,$56.8M


In [7]:
imax = curve.exp_profit.idxmax()
print(f"profit peaks at PD<{curve.pd_cut[imax]:.3f} / approval {curve.approval_rate[imax]:.1%} "
      f"= {money(curve.exp_profit.max())}")
print(f"at 100% approval profit falls to {money(curve.exp_profit.iloc[-1])} "
      f"-> the curve has a real interior optimum")

profit peaks at PD<0.174 / approval 73.2% = $56.8M
at 100% approval profit falls to $3.0M -> the curve has a real interior optimum


## Scenario stress (a single table — the full layer is deferred, spec §1)
Take the book approved under the CURRENT policy (PD<0.15), then stress: multiply every
PD by a factor and re-price *that same population* (no re-underwriting).

In [8]:
LGD = cfg["expected_loss"]["lgd"]
booked = base[(base.pd_hat < 0.15) & (base.fico_mid >= args["fico_min"])].copy()
stress = []
for f in (1.0, 1.2, 1.5):
    b = booked.copy()
    b["pd_hat"] = (b.pd_hat * f).clip(upper=1.0)
    b["expected_loss"] = b.pd_hat * b.ead * LGD
    # approve everyone in the fixed book (cut-off 1.0), keep all other assumptions
    m = approved_metrics(b, 1.0, fico_min=0, cost_of_funds=args["cost_of_funds"],
                         horizon=args["horizon"], amort_factor=args["amort_factor"],
                         servicing_cost=args["servicing_cost"])
    stress.append({"pd_x": f, "mean_pd": round(b.pd_hat.mean(), 3),
                   "exp_loss": money(m["exp_loss"]), "exp_profit": money(m["exp_profit"])})
pd.DataFrame(stress)

,pd_x,mean_pd,exp_loss,exp_profit
0,1.0,0.088,$207.4M,$54.0M
1,1.2,0.105,$248.8M,$-3.4M
2,1.5,0.132,$311.0M,$-89.5M


In [9]:
con.close()